# Week 4: Supervised Learning & Classification
## Notebook 09: Ordinal Logistic Regression & Proportional Odds Model

### 1. Conceptual Intuition & Theoretical Background
When target variables have a natural **ordered sequence** (e.g., Customer Rating: 1-Star < 2-Star < 3-Star < 4-Star < 5-Star; Disease Severity: Mild < Moderate < Severe), treating them as:
- Standard regression assumes equal numerical distance between classes ($5 - 4 \neq 2 - 1$).
- Standard multi-class classification (Softmax) completely discards the natural ordering.

**Ordinal Logistic Regression (Cumulative Logit Model)** introduces a latent continuous variable:
$$Y^* = X\beta + \epsilon, \quad \epsilon \sim \text{Logistic}(0, 1)$$
With strictly ordered cutpoints $\alpha_1 < \alpha_2 < \dots < \alpha_{K-1}$:
$$P(Y \le j | X) = \sigma(\alpha_j - X\beta) = \frac{1}{1 + e^{-(\alpha_j - X\beta)}}$$

The **Proportional Odds Assumption** dictates that the slope $\beta$ remains identical across all cutpoints.

---

### 2. Scratch Python Implementation: Ordinal Proportional Odds

In [ ]:
import numpy as np

class OrdinalLogisticRegressionScratch:
    def __init__(self, n_classes=3, lr=0.05, n_iter=1500):
        self.K = n_classes
        self.lr = lr
        self.n_iter = n_iter
        self.beta = None
        self.alphas = None
        
    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.beta = np.zeros(n_features)
        # Initialize sorted cutpoints
        self.alphas = np.linspace(-1.0, 1.0, self.K - 1)
        
        for epoch in range(self.n_iter):
            # Compute cumulative probabilities: P(Y <= j)
            xb = np.dot(X, self.beta)
            
            # Probabilities per class category
            cum_probs = []
            for alpha in self.alphas:
                cum_probs.append(self._sigmoid(alpha - xb))
            cum_probs = np.column_stack(cum_probs) # shape (N, K-1)
            
            # Category probabilities
            # P(Y=0) = P(Y<=0)
            # P(Y=j) = P(Y<=j) - P(Y<=j-1)
            # P(Y=K-1) = 1 - P(Y<=K-2)
            probs = np.zeros((n_samples, self.K))
            probs[:, 0] = cum_probs[:, 0]
            for j in range(1, self.K - 1):
                probs[:, j] = np.maximum(1e-8, cum_probs[:, j] - cum_probs[:, j - 1])
            probs[:, self.K - 1] = np.maximum(1e-8, 1.0 - cum_probs[:, -1])
            
            # Approximate gradient step via likelihood
            y_one_hot = np.zeros((n_samples, self.K))
            y_one_hot[np.arange(n_samples), y] = 1.0
            
            err = probs - y_one_hot
            grad_beta = np.dot(X.T, err.mean(axis=1)) / n_samples
            self.beta -= self.lr * grad_beta
            
            # Enforce strictly monotonic cutpoints
            for k in range(len(self.alphas) - 1):
                if self.alphas[k] >= self.alphas[k+1]:
                    self.alphas[k+1] = self.alphas[k] + 0.05
                    
        return self
        
    def predict(self, X):
        xb = np.dot(X, self.beta)
        probs = np.zeros((len(X), self.K))
        cum = [self._sigmoid(alpha - xb) for alpha in self.alphas]
        cum = np.column_stack(cum)
        probs[:, 0] = cum[:, 0]
        for j in range(1, self.K - 1):
            probs[:, j] = np.maximum(0.0, cum[:, j] - cum[:, j - 1])
        probs[:, self.K - 1] = np.maximum(0.0, 1.0 - cum[:, -1])
        return np.argmax(probs, axis=1)

# Generate Synthetic Ordinal Clinical Trials Dataset
# Classes: 0 (Mild), 1 (Moderate), 2 (Severe)
np.random.seed(42)
X_clin = np.random.randn(300, 2)
# Latent score
latent_score = 1.8 * X_clin[:, 0] - 1.2 * X_clin[:, 1] + np.random.logistic(0, 1, 300)
y_clin = np.digitize(latent_score, bins=[-0.8, 1.2])

model_ord = OrdinalLogisticRegressionScratch(n_classes=3, lr=0.1, n_iter=1000).fit(X_clin, y_clin)
preds = model_ord.predict(X_clin)

acc = np.mean(preds == y_clin) * 100
print(f"Ordinal Logistic Classifier Accuracy: {acc:.1f}%")
print(f"Fitted Cutpoints: {np.round(model_ord.alphas, 3)}")

### 3. Step-by-Step Code Walkthrough
1. **Cumulative Odds Mapping**: We calculate the probability of falling at or below boundary $j$.
2. **Category Probability Decomposition**: Subtracting consecutive cumulative boundaries yields probability mass for each distinct discrete tier.
3. **Cutpoint Monotonicity**: Preserves structural constraint $\alpha_1 < \alpha_2 < \dots < \alpha_{K-1}$.

---

### 4. Industry Applications & Trade-offs
- **Customer Satisfaction & Net Promoter Score (NPS):** Modeling likelihood of user ratings without distorting distance intervals.
- **Trade-off:** The proportional odds assumption can fail if an explanatory variable has differing effects across low vs high thresholds.

---

### 5. Top Interview Questions & Answers
**Q1: What is the Proportional Odds Assumption in Ordinal Logistic Regression?**
> **Answer:** It assumes the relationship between explanatory variables and the odds of exceeding any given threshold is constant across all thresholds (parallel slopes assumption).

**Q2: Why should an analyst avoid converting ordered ratings into standard integers and fitting Linear Regression?**
> **Answer:** Linear regression assumes identical interval spacing between categories ($1 \to 2$ is identical to $4 \to 5$) and can predict values outside the valid discrete range.